# Primary Demand Forecasting — Training

Model: **CatBoost (MAE, log1p) + inventory-conservation blend**.

Distributor stock balance: `dbr_stock_t = dbr_stock_{t-1} + primary_t - secondary_t`  =>
`primary_t ≈ secondary_t + (dbr_stock_t - dbr_stock_{t-1})`. This `primary_conservation` term (corr 0.905 with primary)
is blended with CatBoost: **final = 0.3·CatBoost + 0.7·conservation**. No leakage.
Saves `../models/primary_model.joblib` (the blend weight is stored in the artifact and applied by `main_2.py`).


In [ ]:
import pandas as pd, numpy as np, math, joblib
from catboost import CatBoostRegressor
from sklearn.metrics import r2_score
import warnings; warnings.filterwarnings('ignore')


### 1. Feature engineering (primary lags + cross-channel + conservation)


In [ ]:
TARGET='primary'; CONS_WEIGHT=0.7
FEATS=['ret_stock','ret_dos','wod','activating_outlet','dbr_stock','dbr_dos','stocking_outlet','secondary','tertiary',
       'seasonality_1(sin)','seasonality_2(cos)','total_stock','activation_ratio','secondary_lag_1','tertiary_lag_1',
       'dbr_change','ret_change','primary_conservation',
       'primary_lag_1','primary_lag_2','primary_lag_3','primary_lag_4','primary_lag_6','primary_lag_8',
       'primary_roll_mean_3','primary_roll_mean_4','primary_roll_mean_8','primary_roll_std_4','primary_roll_max_4','primary_diff_lag']
LOGF=['ret_stock','ret_dos','wod','activating_outlet','dbr_stock','dbr_dos','stocking_outlet','secondary','tertiary','total_stock',
      'secondary_lag_1','tertiary_lag_1','primary_lag_1','primary_lag_2','primary_lag_3','primary_lag_4','primary_lag_6','primary_lag_8',
      'primary_roll_mean_3','primary_roll_mean_4','primary_roll_mean_8','primary_roll_max_4']

def seasonality(df):
    o=df.copy(); a=2*math.pi*pd.to_numeric(o['week'],errors='coerce').fillna(0)/59.0
    o['seasonality_1(sin)']=np.sin(a); o['seasonality_2(cos)']=np.cos(a); return o

def add_features(df):
    o=df.copy()
    sk=['model_family']+(['model_code'] if 'model_code' in o else [])+(['year'] if 'year' in o else [])+(['_source_rank'] if '_source_rank' in o else [])+['week']
    o=o.sort_values(sk).reset_index(drop=True)
    gp=o.groupby(['model_family'],sort=False); g=gp[TARGET]
    for l in (1,2,3,4,6,8): o[f'primary_lag_{l}']=g.shift(l)
    sh=g.shift(1)
    for w in (3,4,8): o[f'primary_roll_mean_{w}']=sh.rolling(w,min_periods=w).mean()
    o['primary_roll_std_4']=sh.rolling(4,min_periods=4).std(); o['primary_roll_max_4']=sh.rolling(4,min_periods=4).max()
    o['primary_diff_lag']=o['primary_lag_1']-o['primary_lag_2']
    o['secondary_lag_1']=gp['secondary'].shift(1); o['tertiary_lag_1']=gp['tertiary'].shift(1)
    o['total_stock']=pd.to_numeric(o['ret_stock'],errors='coerce').fillna(0)+pd.to_numeric(o['dbr_stock'],errors='coerce').fillna(0)
    o['activation_ratio']=pd.to_numeric(o['activating_outlet'],errors='coerce').fillna(0)/(pd.to_numeric(o['stocking_outlet'],errors='coerce').fillna(0)+1e-5)
    o['dbr_stock_lag_1']=gp['dbr_stock'].shift(1); o['dbr_change']=pd.to_numeric(o['dbr_stock'],errors='coerce')-o['dbr_stock_lag_1']
    o['ret_stock_lag_1']=gp['ret_stock'].shift(1); o['ret_change']=pd.to_numeric(o['ret_stock'],errors='coerce')-o['ret_stock_lag_1']
    o['primary_conservation']=pd.to_numeric(o['secondary'],errors='coerce').fillna(0)+o['dbr_change'].fillna(0)
    return o

def build_X(df):
    o=df.copy()
    for c in FEATS:
        if c not in o: o[c]=0.0
    X=o[FEATS].apply(pd.to_numeric,errors='coerce').fillna(0)
    for c in LOGF:
        if c in X: X[c]=np.log1p(np.clip(X[c],0,None))
    return X.replace([np.inf,-np.inf],np.nan).fillna(0)


### 2. Train CatBoost on cleaned_primary_final.csv


In [ ]:
csv=pd.read_csv('cleaned_primary_final.csv')
csv['start_date']=pd.to_datetime(csv['start_date']); csv['year']=csv['start_date'].dt.year
csv=csv[csv[TARGET]>=0].copy()
csv=add_features(seasonality(csv)).dropna(subset=[f'primary_lag_{l}' for l in (1,2,3,4,6,8)]).reset_index(drop=True)
model=CatBoostRegressor(iterations=800,learning_rate=0.05,depth=6,loss_function='MAE',random_seed=42,verbose=0,allow_writing_files=False)
model.fit(build_X(csv), np.log1p(np.clip(csv[TARGET].values,0,None)))
print('Trained on', len(csv), 'rows')


### 3. Evaluate with conservation blend (history 41-48 + test 49-59)


In [ ]:
def acc(a,b):
    a=np.asarray(a); b=np.clip(np.asarray(b),0,None)
    return max(0,100-np.mean(np.abs((a-b)/np.clip(np.abs(a),1,None))*100))

csvf=pd.read_csv('cleaned_primary_final.csv'); hero=csvf[csvf['model_code']=='HEROSHAK2025H11']
base=csvf[csvf['model_code']!='HEROSHAK2025H11'].copy(); base['is_test']=False
h=pd.read_excel('HEROSHAK2025H11-Primary-Lag.xlsx'); h['is_test']=False
t=pd.read_excel('HEROSHAK2025H11-Primary-test.xlsx'); t['is_test']=True
for d in (h,t): d['start_date']=d['week'].map(hero.set_index('week')['start_date'].to_dict())
comb=pd.concat([base,h,t],ignore_index=True)
comb['start_date']=pd.to_datetime(comb['start_date']); comb['year']=comb['start_date'].dt.year
comb=add_features(seasonality(comb)).dropna(subset=[f'primary_lag_{l}' for l in (1,2,3,4,6,8)]).reset_index(drop=True)
sc=comb[comb['is_test']==True].copy(); yt=sc[TARGET].values; wk=sc['week'].values
pcb=np.clip(np.expm1(model.predict(build_X(sc))),0,None)
cons=np.clip(sc['primary_conservation'].values,0,None)
pred=(1-CONS_WEIGHT)*pcb + CONS_WEIGHT*cons
print(f'CatBoost only      : {acc(yt,pcb):.2f}%')
print(f'Conservation only  : {acc(yt,cons):.2f}%')
print(f'Blend (final)      : {acc(yt,pred):.2f}%  R2={r2_score(yt,pred):.3f}')
display(pd.DataFrame({'Week':wk,'Actual':yt,'Pred':np.round(pred,0),'APE%':np.round(np.abs(yt-pred)/np.clip(yt,1,None)*100,1)}))


### 4. Save model artifact (with conservation blend config)


In [ ]:
import os; os.makedirs('../models',exist_ok=True)
joblib.dump({'model':model,'features':FEATS,'log_feature_cols':LOGF,'target':'primary','target_log1p':True,'scaler':None,
             'conservation_col':'primary_conservation','conservation_weight':CONS_WEIGHT,
             'model_version':'CatBoost_primary_conservation_v1'},
            '../models/primary_model.joblib')
print('Saved ../models/primary_model.joblib')
